#### **Imported Libraries:**

In [1]:
import pygame
import random
from collections import deque


pygame 2.6.1 (SDL 2.28.4, Python 3.13.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


##### **Game Sprits:**

In [2]:
def load_images():
    images = {}
    images["green"] = pygame.image.load("assets/green_grass.png")
    images["light"] = pygame.image.load("assets/light_grass.png")
    images["white_chicken"] = pygame.image.load("assets/white_chicken.png")
    images["black_chicken"] = pygame.image.load("assets/black_chicken.png")
    images["white_egg"] = pygame.image.load("assets/white_egg.png")
    images["black_egg"] = pygame.image.load("assets/dark_egg.png")
    images["trap"] = pygame.image.load("assets/trap_hole.webp")
    images["turd"] = pygame.image.load("assets/chick_turd.webp")

    return images

##### **Game Screen Constants:**

In [3]:
WIDTH = 800
HEIGHT = 600

ROWS = 8
COLS = 8

TILE_SIZE = 64

In [4]:
MOVES = {
    pygame.K_UP: (-1, 0),
    pygame.K_DOWN: (1, 0),
    pygame.K_LEFT: (0, -1),
    pygame.K_RIGHT: (0, 1)
}

##### **Game Board Drawing Utility:**

In [5]:
def draw_board(screen, images):
    for row in range(ROWS):
        for col in range(COLS):
            x = col * TILE_SIZE
            y = row * TILE_SIZE

            if (row + col) % 2 == 0:
                tile = images["green"]
            else:
                tile = images["light"]

            tile = pygame.transform.scale(tile, (TILE_SIZE, TILE_SIZE))
            screen.blit(tile, (x, y))

In [6]:
def reachable_tiles(start, traps):
    visited = set()
    queue = deque([start])

    while queue:
        r, c = queue.popleft()

        if (r, c) in visited:
            continue
        if (r, c) in traps:
            continue

        visited.add((r, c))

        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < ROWS and 0 <= nc < COLS:
                queue.append((nr, nc))

    return visited

In [7]:
def generate_trapdoors(num_traps=12):
    import random

    forbidden = {
        (0, 0), (0, 1), (1, 0),
        (7, 7), (7, 6), (6, 7)
    }

    while True:
        traps = set()

        while len(traps) < num_traps:
            r = random.randint(0, ROWS - 1)
            c = random.randint(0, COLS - 1)

            if (r, c) not in forbidden:
                traps.add((r, c))

        white_reach = reachable_tiles((0, 0), traps)
        black_reach = reachable_tiles((7, 7), traps)

        MIN_ACCESSIBLE = 15  # tweak if needed

        if len(white_reach) >= MIN_ACCESSIBLE and len(black_reach) >= MIN_ACCESSIBLE:
            return traps

In [8]:
def draw_traps(screen, state, images):
    for (r, c) in state.revealed_traps:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["trap"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [9]:
def draw_eggs(screen, state, images):
    for (r, c), player_id in state.eggs.items():
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img_key = "white_egg" if player_id == 0 else "black_egg"
        img = pygame.transform.scale(images[img_key], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [10]:
def draw_turds(screen, state, images):
    for (r, c) in state.turds:
        x = c * TILE_SIZE
        y = r * TILE_SIZE

        img = pygame.transform.scale(images["turd"], (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [11]:
def draw_ui(screen, player1, player2, font, state):
    x_offset = COLS * TILE_SIZE + 20

    texts = [
        f"White Eggs: {player1.eggs_placed}",
        f"White Moves: {player1.moves_left}",
        f"White Turds: {player1.turds_remaining}",
        "",
        f"Black Eggs: {player2.eggs_placed}",
        f"Black Moves: {player2.moves_left}",
        f"Black Turds: {player2.turds_remaining}",
        "",
    ]

    current = get_current_player(state)
    turn_text = "White Turn" if current == player1 else "Black Turn"
    texts.append(turn_text)

    y = 50
    for text in texts:
        label = font.render(text, True, (0, 0, 0))
        screen.blit(label, (x_offset, y))
        y += 30

##### **Player Class and Functions:**

In [12]:
class Player:
    def __init__(self, row, col, image):
        self.row = row
        self.col = col
        self.image = image
        self.prev_row = row
        self.prev_col = col

        self.moves_left = 30
        self.turds_remaining = 4
        self.eggs_placed = 0

In [13]:
def draw_players(screen, players):
    for player in players:
        x = player.col * TILE_SIZE
        y = player.row * TILE_SIZE

        img = pygame.transform.scale(player.image, (TILE_SIZE, TILE_SIZE))
        screen.blit(img, (x, y))

In [14]:
def update_previous_position(player):
    player.prev_row = player.row
    player.prev_col = player.col

In [15]:
def get_current_player(state):
    return state.players[state.current_player_index]

In [16]:
def switch_turn(state):
    state.current_player_index = 1 - state.current_player_index

In [17]:
def move_player(state, direction):
    player = get_current_player(state)

    if player.moves_left <= 0:
        return "NO_MOVES"

    dr, dc = direction
    new_r = player.row + dr
    new_c = player.col + dc

    if not is_valid_move(state, player, new_r, new_c):
        return "ILLEGAL"

    # save previous
    update_previous_position(player)

    # move
    player.row = new_r
    player.col = new_c

    # resolve tile
    result = resolve_tile(state, player)

    if result == "TRAP":
        player.moves_left -= 1  
        switch_turn(state)
        return "TRAP"

    return "SAFE"

In [18]:
def place_egg(state, player):
    pos = (player.row, player.col)

    if pos in state.eggs:
        return False
    if pos in state.turds:
        return False
    if pos in state.revealed_traps:
        return False

    state.eggs[pos] = state.current_player_index
    player.eggs_placed += 1

    return True

In [19]:
def place_turd(state, player):
    pos = (player.row, player.col)

    if player.turds_remaining <= 0:
        return False
    if pos in state.eggs or pos in state.turds:
        return False

    state.turds.add(pos)
    player.turds_remaining -= 1

    return True

##### **Game Logic Utility:**

In [20]:
def handle_action_phase(state, player, event):
    if event.key == pygame.K_e:
        success = place_egg(state, player)

    elif event.key == pygame.K_t:
        success = place_turd(state, player)

    elif event.key == pygame.K_SPACE:
        success = True

    else:
        return  # wait for valid action key

    if success:
        player.moves_left -= 1
        switch_turn(state)

In [21]:
def is_valid_move(state, player, new_r, new_c):
    # outside board
    if not (0 <= new_r < ROWS and 0 <= new_c < COLS):
        return False

    # opponent position
    opponent = state.players[1 - state.current_player_index]
    if (new_r, new_c) == (opponent.row, opponent.col):
        return False

    # turd
    if (new_r, new_c) in state.turds:
        return False

    # revealed trap
    if (new_r, new_c) in state.revealed_traps:
        return False

    return True

In [22]:
def handle_input(state, event):
    if event.key in MOVES:
        result = move_player(state, MOVES[event.key])

        if result == "ILLEGAL":
            return

        if result == "TRAP":
            return

        # WAIT for action input → DO NOT auto switch turn
        state.awaiting_action = True
        return

    # 🔥 Action phase
    if state.awaiting_action:
        player = get_current_player(state)
        handle_action_phase(state, player, event)
        state.awaiting_action = False

In [23]:
def resolve_tile(state, player):
    pos = (player.row, player.col)

    # Trapdoor
    if pos in state.traps:
        state.revealed_traps.add(pos)

        # penalty
        player.moves_left -= 2

        # return back
        player.row = player.prev_row
        player.col = player.prev_col

        return "TRAP"

    return "SAFE"

In [24]:
class GameState:
    def __init__(self, player1, player2, traps):
        self.players = [player1, player2]
        self.current_player_index = 0

        self.traps = traps
        self.revealed_traps = set()

        self.turds = set()
        self.eggs = {}  # (row, col) → player_id

        self.awaiting_action = False

In [25]:
pygame.init()

screen = pygame.display.set_mode((1000, 600))
pygame.display.set_caption("Eggplosion")

images = load_images()
font = pygame.font.SysFont(None, 24)

player1 = Player(0, 0, images["white_chicken"])
player2 = Player(7, 7, images["black_chicken"])

traps = generate_trapdoors()
state = GameState(player1, player2, traps)

players = [player1, player2]

running = True
clock = pygame.time.Clock()

while running:
    clock.tick(60)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        if event.type == pygame.KEYDOWN:
            handle_input(state, event)

    screen.fill((255, 255, 255))

    draw_board(screen, images)
    draw_traps(screen, state, images)
    draw_eggs(screen, state, images)
    draw_turds(screen, state, images)
    draw_players(screen, state.players)
    draw_ui(screen, player1, player2, font, state)

    pygame.display.flip()

pygame.quit()
